# Prep for website
Form the same as the kelp outputs
* Split out kelp geometries
* Array of named locations and info for satellite images

In [ ]:
import geopandas
import leafmap
import pandas
import pathlib
import dask.distributed
import datetime

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Value to edit

In [ ]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                       "Purau", "Ihutai", "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Ihutai", "LeftBank_Nov25", "ThePoint_Nov25"]

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .80
model_path = utils.get_models_path(sample_method=sample_method, method_2_threshold=method_2_threshold)
model_file = model_path / "sample_purity_80_test_on_Ihutai_and_train_on_all_sites_grouped_classes.joblib"
prediction_sites = large_seagrass_sites
years = [2023, 2024, 2025]

# Cells to run
* Loop over prediction sites and years
* Split out seagrass extents for each date
* Save the identifying satellite information for each date

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
max_cloud_cover = 0

In [ ]:
for prediction_site in prediction_sites:
    output_folder = data_path / "website" / prediction_site
    output_folder.mkdir(exist_ok=True, parents=True)

    if (output_folder / "presence_absence_map.gpkg").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    seagrass_extents_all_years = []
    info = {"ids": [], "percentages_2":[], "percentage_98":[], "filename":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
    
        seagrass_file = data_path / "predictions" / "predictions" / f"{prediction_site}_{year}_prediction_{model_file.stem}_seagrass.gpkg"
    
        seagrass_extents = geopandas.read_file(seagrass_file)
        seagrass_extents_all_years.append(seagrass_extents)
        # Look up the satellite tile information
        
        for index, row in seagrass_extents.iterrows():
            seagrass_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
            date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
    
            filename = f"{date_YYMMDD}_seagrass.gpkg"
            info["filename"].append(filename)
            seagrass_extent.to_file(output_folder / filename)
    
            # Get Satellite tile information
            #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
            site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
            tile_id, percentage_2, percentage_98 = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
            
            info["ids"].append(tile_id)
            info["percentages_2"].append(percentage_2)
            info["percentage_98"].append(percentage_98); 

    seagrass_extents = pandas.concat(seagrass_extents_all_years, ignore_index=True)
    seagrass_extents['area'] = seagrass_extents.area
    seagrass_extents = pandas.concat([seagrass_extents, pandas.DataFrame(info)], axis=1)
    seagrass_extents = seagrass_extents.rename(columns={"ids": "Satellite Tile IDs", "percentages_2": "Percentile 2", "percentage_98": "Percentile 98"})
    
    seagrass_extents.drop(columns='geometry').to_csv(output_folder / "info_all_dates.csv", index=False)
    seagrass_extents.dissolve()[["geometry"]].to_file(output_folder / "presence_absence_map.gpkg")
